In [ ]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import numpy as np

## Define nodes

In [34]:
# Define nodes
station_codes = pd.read_csv('mrt_lrt_stations.csv')
# exclude all the stations that LINE_COLOR is not grey
nodes = station_codes[station_codes['LINE_COLOR'] != 'Grey']['ALPHANUMERIC_CODE'].unique().tolist()
nodes.insert(nodes.index('EW33') + 1, 'CG0') # insert CG0 because tanah merah has no CG code but it is an interchange to go to expo/CG
nodes.insert(nodes.index('CC29') + 1, 'CE0') # insert CE0 because promenade has no CE code but it is an interchange to go to sentosa/CE
# nodes

### Pull only the downtown line, i.e. starting with 'DT'


In [37]:
# nodes is a list.
# print(type(nodes))
dtl_stations = [station for station in nodes if station.startswith('DT')]
dtl_stations

['DT1',
 'DT2',
 'DT3',
 'DT4',
 'DT5',
 'DT6',
 'DT7',
 'DT8',
 'DT9',
 'DT10',
 'DT11',
 'DT12',
 'DT13',
 'DT14',
 'DT15',
 'DT16',
 'DT17',
 'DT18',
 'DT19',
 'DT20',
 'DT21',
 'DT22',
 'DT23',
 'DT24',
 'DT25',
 'DT26',
 'DT27',
 'DT28',
 'DT29',
 'DT30',
 'DT31',
 'DT32',
 'DT33',
 'DT34',
 'DT35']

## Define arcs

In [38]:
# each node is connected to the next node in the same line, and also to the interchanges
# the nodes have values e.g. NS1, NS2, NS3. NS1 has an arc to NS2, NS2 has an arc to NS3 but NS1 has no arc to NS3.
arcs = []
for i in range(len(nodes) - 1):
    if nodes[i][:2] == nodes[i + 1][:2]:
        arcs.append((nodes[i], nodes[i + 1]))
# print(arcs)
interchanges = [('NS1', 'EW24'), ('NS9', 'TE2'), ('CE0', 'CC4'), ('CE0', 'DT16'),
                ('NS17', 'CC15'), ('NS21', 'DT11'), ('NS22', 'TE14'), 
               ('NS24', 'NE6'), ('NS24', 'CC1'), ('NS25', 'EW13'), 
               ('NS26', 'EW14'), ('NS27', 'TE20'), ('NS27', 'CE2'),
               ('CC22', 'EW21'), ('EW16', 'NE3'), ('EW16', 'TE17'), 
               ('EW12', 'DT14'), ('EW8', 'CC9'), ('EW4', 'CG0'), 
               ('EW2', 'DT32'), ('CG1', 'DT35'), ('CC19', 'DT9'),
               ('DT10', 'TE11'), ('NE7', 'DT12'), ('DT15', 'CC4'),
               ('DT16', 'CE1'), ('DT19', 'NE4'), ('DT26', 'CC10'),
               ('CC29', 'NE1'), ('CC17', 'TE9'), ('CC13', 'NE12'),
               ('CC1', 'NE6'), ('CE2', 'TE20'), ('NE3', 'TE17'), 
               ]
for interchange in interchanges:
    arcs.append(interchange)
# since all arcs are bidirectional, need to include the other direction as well.
for arc in arcs.copy():
    arcs.append((arc[1], arc[0]))

print(arcs)

[('NS1', 'NS2'), ('NS2', 'NS3'), ('NS3', 'NS4'), ('NS4', 'NS5'), ('NS5', 'NS7'), ('NS7', 'NS8'), ('NS8', 'NS9'), ('NS9', 'NS10'), ('NS10', 'NS11'), ('NS11', 'NS12'), ('NS12', 'NS13'), ('NS13', 'NS14'), ('NS14', 'NS15'), ('NS15', 'NS16'), ('NS16', 'NS17'), ('NS17', 'NS18'), ('NS18', 'NS19'), ('NS19', 'NS20'), ('NS20', 'NS21'), ('NS21', 'NS22'), ('NS22', 'NS23'), ('NS23', 'NS24'), ('NS24', 'NS25'), ('NS25', 'NS26'), ('NS26', 'NS27'), ('NS27', 'NS28'), ('EW1', 'EW2'), ('EW2', 'EW3'), ('EW3', 'EW4'), ('EW4', 'EW5'), ('EW5', 'EW6'), ('EW6', 'EW7'), ('EW7', 'EW8'), ('EW8', 'EW9'), ('EW9', 'EW10'), ('EW10', 'EW11'), ('EW11', 'EW12'), ('EW12', 'EW13'), ('EW13', 'EW14'), ('EW14', 'EW15'), ('EW15', 'EW16'), ('EW16', 'EW17'), ('EW17', 'EW18'), ('EW18', 'EW19'), ('EW19', 'EW20'), ('EW20', 'EW21'), ('EW21', 'EW22'), ('EW22', 'EW23'), ('EW23', 'EW24'), ('EW24', 'EW25'), ('EW25', 'EW26'), ('EW26', 'EW27'), ('EW27', 'EW28'), ('EW28', 'EW29'), ('EW29', 'EW30'), ('EW30', 'EW31'), ('EW31', 'EW32'), ('EW3

#### Pull the downtown line arcs

In [40]:
dtl_arcs = [arc for arc in arcs if arc[0] in dtl_stations and arc[1] in dtl_stations]
dtl_arcs

[('DT1', 'DT2'),
 ('DT2', 'DT3'),
 ('DT3', 'DT4'),
 ('DT4', 'DT5'),
 ('DT5', 'DT6'),
 ('DT6', 'DT7'),
 ('DT7', 'DT8'),
 ('DT8', 'DT9'),
 ('DT9', 'DT10'),
 ('DT10', 'DT11'),
 ('DT11', 'DT12'),
 ('DT12', 'DT13'),
 ('DT13', 'DT14'),
 ('DT14', 'DT15'),
 ('DT15', 'DT16'),
 ('DT16', 'DT17'),
 ('DT17', 'DT18'),
 ('DT18', 'DT19'),
 ('DT19', 'DT20'),
 ('DT20', 'DT21'),
 ('DT21', 'DT22'),
 ('DT22', 'DT23'),
 ('DT23', 'DT24'),
 ('DT24', 'DT25'),
 ('DT25', 'DT26'),
 ('DT26', 'DT27'),
 ('DT27', 'DT28'),
 ('DT28', 'DT29'),
 ('DT29', 'DT30'),
 ('DT30', 'DT31'),
 ('DT31', 'DT32'),
 ('DT32', 'DT33'),
 ('DT33', 'DT34'),
 ('DT34', 'DT35'),
 ('DT2', 'DT1'),
 ('DT3', 'DT2'),
 ('DT4', 'DT3'),
 ('DT5', 'DT4'),
 ('DT6', 'DT5'),
 ('DT7', 'DT6'),
 ('DT8', 'DT7'),
 ('DT9', 'DT8'),
 ('DT10', 'DT9'),
 ('DT11', 'DT10'),
 ('DT12', 'DT11'),
 ('DT13', 'DT12'),
 ('DT14', 'DT13'),
 ('DT15', 'DT14'),
 ('DT16', 'DT15'),
 ('DT17', 'DT16'),
 ('DT18', 'DT17'),
 ('DT19', 'DT18'),
 ('DT20', 'DT19'),
 ('DT21', 'DT20'),
 ('DT22'

## Define costs
* Taken from the dataset that has the timings between stations, including transfer timings
* Link: https://docs.google.com/spreadsheets/d/13CTjSiDdrjvIJvvCso10TWSP5Bf1kZrR77W638BVFkI/edit?gid=987939193#gid=987939193

In [22]:
station_timings_cost = pd.read_csv('station_costs_no_interchanges.csv')
# Map each station to cost given the codes
costs = {}
# print(station_timings_cost.iloc[2]['station_code'])
for index, row in station_timings_cost.iterrows():
    station_code = row['station_code']
    # convert travel_time to a string
    row['travel_time'] = str(row['travel_time'])
    prev_station_code = station_timings_cost.iloc[index-1]['station_code'] if index >= 0 else None
    # convert travel_time from 0:04:25 to number of seconds, which is 4*60 + 25 = 265 seconds
    cost_in_seconds = row['travel_time'].split(':') if row['travel_time'] != '0:00:00' else [0, 0, 0]
    cost_in_seconds = int(cost_in_seconds[0]) * 3600 + (int(cost_in_seconds[1]) * 60 + int(cost_in_seconds[2]))
    if prev_station_code and station_code[:2] == prev_station_code[:2] and cost_in_seconds != 0: # if the station is on the same line as the previous station, then the cost is the travel time between them
        costs[(prev_station_code, station_code)] = cost_in_seconds
        costs[(station_code, prev_station_code)] = cost_in_seconds # since the arcs are bidirectional, the cost is the same in both directions
# print(costs)
# need to manually add the costs for transfer at interchanges
station_timings_cost_interchanges = pd.read_csv('station_costs_interchanges.csv')
station_timings_cost_interchanges = station_timings_cost_interchanges.dropna()
for index, row in station_timings_cost_interchanges.iterrows():
    station_code_1 = row['station_code_1']
    station_code_2 = row['station_code_2']
    # convert travel_time to a string
    row['travel_time'] = str(row['travel_time'])
    cost_in_seconds = row['travel_time'].split(':') if row['travel_time'] != '0:00:00' else [0, 0, 0]
    cost_in_seconds = int(cost_in_seconds[0]) * 3600 + (int(cost_in_seconds[1]) * 60 + int(cost_in_seconds[2]))
    costs[(station_code_1, station_code_2)] = cost_in_seconds
    costs[(station_code_2, station_code_1)] = cost_in_seconds # since the arcs are bidirectional, the cost is the same in both directions

print(costs)

{('TE1', 'TE2'): 116, ('TE2', 'TE1'): 116, ('TE2', 'TE3'): 152, ('TE3', 'TE2'): 152, ('TE3', 'TE4'): 265, ('TE4', 'TE3'): 265, ('TE4', 'TE5'): 183, ('TE5', 'TE4'): 183, ('TE5', 'TE6'): 139, ('TE6', 'TE5'): 139, ('TE6', 'TE7'): 129, ('TE7', 'TE6'): 129, ('TE7', 'TE8'): 164, ('TE8', 'TE7'): 164, ('TE8', 'TE9'): 200, ('TE9', 'TE8'): 200, ('TE9', 'TE11'): 217, ('TE11', 'TE9'): 217, ('TE11', 'TE12'): 190, ('TE12', 'TE11'): 190, ('TE12', 'TE13'): 122, ('TE13', 'TE12'): 122, ('TE13', 'TE14'): 128, ('TE14', 'TE13'): 128, ('TE14', 'TE15'): 142, ('TE15', 'TE14'): 142, ('TE15', 'TE16'): 90, ('TE16', 'TE15'): 90, ('TE16', 'TE17'): 112, ('TE17', 'TE16'): 112, ('TE17', 'TE18'): 102, ('TE18', 'TE17'): 102, ('TE18', 'TE19'): 119, ('TE19', 'TE18'): 119, ('TE19', 'TE20'): 109, ('TE20', 'TE19'): 109, ('TE20', 'TE22'): 164, ('TE22', 'TE20'): 164, ('TE22', 'TE23'): 198, ('TE23', 'TE22'): 198, ('TE23', 'TE24'): 125, ('TE24', 'TE23'): 125, ('TE24', 'TE25'): 132, ('TE25', 'TE24'): 132, ('TE25', 'TE26'): 113, 

#### Pull the downtown line costs

In [51]:
# a single entry in the cost dictionary is ('TE1', 'TE2'): 116
# take a subset of those costs whose keys are in dtl_arcs, i.e. both stations are in dtl_stations
dtl_costs = {arc: cost for arc, cost in costs.items() if arc in dtl_arcs}
dtl_costs

{('DT35', 'DT34'): 81,
 ('DT34', 'DT35'): 81,
 ('DT34', 'DT33'): 210,
 ('DT33', 'DT34'): 210,
 ('DT33', 'DT32'): 124,
 ('DT32', 'DT33'): 124,
 ('DT32', 'DT31'): 144,
 ('DT31', 'DT32'): 144,
 ('DT31', 'DT30'): 143,
 ('DT30', 'DT31'): 143,
 ('DT30', 'DT29'): 142,
 ('DT29', 'DT30'): 142,
 ('DT29', 'DT28'): 130,
 ('DT28', 'DT29'): 130,
 ('DT28', 'DT27'): 99,
 ('DT27', 'DT28'): 99,
 ('DT27', 'DT26'): 106,
 ('DT26', 'DT27'): 106,
 ('DT26', 'DT25'): 130,
 ('DT25', 'DT26'): 130,
 ('DT25', 'DT24'): 142,
 ('DT24', 'DT25'): 142,
 ('DT24', 'DT23'): 127,
 ('DT23', 'DT24'): 127,
 ('DT23', 'DT22'): 116,
 ('DT22', 'DT23'): 116,
 ('DT22', 'DT21'): 97,
 ('DT21', 'DT22'): 97,
 ('DT21', 'DT20'): 106,
 ('DT20', 'DT21'): 106,
 ('DT20', 'DT19'): 131,
 ('DT19', 'DT20'): 131,
 ('DT19', 'DT18'): 97,
 ('DT18', 'DT19'): 97,
 ('DT18', 'DT17'): 91,
 ('DT17', 'DT18'): 91,
 ('DT17', 'DT16'): 93,
 ('DT16', 'DT17'): 93,
 ('DT16', 'DT15'): 131,
 ('DT15', 'DT16'): 131,
 ('DT15', 'DT14'): 118,
 ('DT14', 'DT15'): 118,
 ('D

#### Create the base model with decision variables

In [54]:
print(len(dtl_stations))

35


In [63]:
TRAIN_COUNT = 100 # can be modified later
m = gp.Model('minimize total passenger travel time + waiting penalty + operating costs')
STATION_COUNT = len(dtl_stations)
# create a TRAIN_COUNT by STATION_COUNT matrix of binary decision variables, where x[i][j] = 1 if train j stops at station i, and 0 otherwise
x = [[0 for j in range(STATION_COUNT)] for i in range(TRAIN_COUNT)]
# Create decision variables for each station and each train, where x[i][j] = 1 if train j stops at station i, and 0 otherwise
for i in range(TRAIN_COUNT):
    for j in range(STATION_COUNT):
        x[i][j] = m.addVar(vtype=GRB.BINARY, name=f'x_{i}_{j}')

y = [0 for j in range(TRAIN_COUNT)]       
# y[j] is binary, 1 if train j is express, 0 if train j is regular
for j in range(TRAIN_COUNT):
    y[j] = m.addVar(vtype=GRB.BINARY, name=f'y_{j}')
z = [0 for j in range(STATION_COUNT)]
# z[j] is binary, 1 if station j is designated as express station, 0 otherwise
for j in range(STATION_COUNT):
    z[j] = m.addVar(vtype=GRB.BINARY, name=f'z_{j}')

u = [0 for j in range(STATION_COUNT)]
# u[j] is binary, measuring how much station j is under-served compared to an arbitrary desired level
# easier comparison: how much cakes is underproduced by the baker compared to the required demand?
# if uj = 0, the station gets enough stopping trains. if uj > 0, the station is getting too 
# few stopping trains and will pay a penalty
for j in range(STATION_COUNT):
    u[j] = m.addVar(vtype=GRB.CONTINUOUS, name=f'u_{j}', lb=0)

print(len(x[0]))

35


#### Constraint 1: terminal stations must always be served and cannot be skipped.

In [65]:
# station DT1 and DT35 must be served by all trains, express or regular, and cannot be skipped.

for t in range(TRAIN_COUNT):
    m.addConstr(x[t][0] == 1)
    m.addConstr(x[t][STATION_COUNT - 1] == 1)

#### If a train is local, it MUST stop at every station.

In [66]:
# add constraint that says if a train is local, it MUST stop at every station, i.e. xtj >= 1 - yt for all t, for all j

for t in range(TRAIN_COUNT):
    for j in range(STATION_COUNT):
        m.addConstr(x[t][j] >= 1 - y[t]) 

#### All stations that have an interchange and is located around the CBD must be served

In [67]:
# Let M be the set of mandatory stop. this can include interchanges, or CBD stations. TODO.
M = []


for t in range(TRAIN_COUNT):
    for station in M:
        m.addConstr(x[t][nodes.index(station)] == 1)


#### Limit the number of express trains possible in a given set of trains

In [68]:
# Let K be the max number of express trains, where K < TRAIN_COUNT. TODO.
K = 20

m.addConstr(gp.quicksum(y[j] for j in range(TRAIN_COUNT)) <= K)

<gurobi.Constr *Awaiting Model Update*>

#### Express trains can only stop at designated express stations, i.e. if train t is express it can ONLY stop where the station is express, else if local train, no restriction

In [69]:
# xtj ≤ zj + (1 – yt) ∀ t ∈ T, j ∈ S
for t in range(TRAIN_COUNT):
    for j in range(STATION_COUNT):
        m.addConstr(x[t][j] <= z[j] + (1 - y[t]))

#### Limit how many stations an express train can skip, if not it may skip the whole thing

In [70]:
# ∑(1-xtj) <= Lyt , ∀ t ∈ T. L is max skipped stations

for t in range(TRAIN_COUNT):
    m.addConstr(gp.quicksum(1 - x[t][j] for j in range(STATION_COUNT)) <= 10 * y[t]) 
    # if a train is express, it can skip at most 10 stations. if a train is local, it can skip 0 stations.
    # check this later. TODO.    

#### Service coverage, i.e. station should ideally receive at least xxxx amount of stopping trains. If not, the objective will penalize per-unit. Let xxxx for each station be Hj, the ideal number of trains to stop at station j. Hj should be a set of numbers

In [71]:
H = [1 for _ in range(STATION_COUNT)]  # Example values, replace with actual ideal numbers

for j in range(STATION_COUNT):
    m.addConstr(gp.quicksum(x[t][j] for t in range(TRAIN_COUNT)) + u[j] >= H[j]) 
    # if the number of trains stopping at station j is less than the ideal number H[j], 
    # then uj will be positive and will pay a penalty in the objective function. if the number of trains stopping at station j is greater than or equal to H[j], then uj will be 0 and there will be no penalty in the objective function.

#### Let ft is the additional operating cost if train t is express. Budget cap, e.g. for extra signage, passenger info, timetable redesign, operational complexity

In [72]:
# let F[t] be additional operating cost if train t is express.
# we have a budget cap, e.g. for extra signage, timetable etc. TODO.

F = [100 for _ in range(TRAIN_COUNT)]  # Example values, replace with actual costs
BUDGET_CONSTRAINT = 5000 # Example value, replace with actual budget constraint
m.addConstr(gp.quicksum(F[t] * y[t] for t in range(TRAIN_COUNT)) <= BUDGET_CONSTRAINT)


<gurobi.Constr *Awaiting Model Update*>

#### Set objective and optimize

In [73]:
# First term: "If train t stops at station j, the system pays a passenger travel-time cost equal to fixed stop time k times passenger impact qj." And xtj is the binary decision variable described above.
# k: Travel-time coefficient, which is a fixed constant time penalty per stop, as we know, when a train stops at a station, there is extra travel time due to deceleration, acceleration and dwell time. k basically sums of all that costs.
# qj: passenger's weight associated with stopping at station j. Represents how many passengers are affected when a train stops at station j. A stop at a busier part of the line may delay more onboard passengers, so qj may be larger there
# Second term: if a station gets fewer stopping trains than desired, we pay a penalty. Prevents the model from making it skip everything and harm all the passengers at the skipped stations. pj is the waiting penalty per unit of under-service at station j. Uj is the amount of service shortfall at station j
# pj is the penalty per unit of under-service at station j, like per-unit underproduction costs of baking a cake. Larger pj nearer to the CBD as that means the station is important, under-serving it is bad. Smaller pj means model is more willing to reduce service there.
# uj is described above, as the amount of units that is underproduced.
# Third term: ft is the additional operating cost if train t is express, yt is defined above.
# Running express trains can create extra cost due to scheduling complexity, operations planning, extra signage, platform information, passenger confusion. Basically non-quantifiable cost
p = [10 for _ in range(STATION_COUNT)] # Example values, replace with actual penalties
q = [1 for _ in range(STATION_COUNT)] # Example values, replace with actual passenger impact weights
k = 1 # Example value, replace with actual travel-time coefficient
m.setObjective(
    k * gp.quicksum(q[j] * x[t][j] for t in range(TRAIN_COUNT) for j in range(STATION_COUNT)) + 
    gp.quicksum(p[j] * u[j] for j in range(STATION_COUNT)) + 
    gp.quicksum(F[t] * y[t] for t in range(TRAIN_COUNT)),
    GRB.MINIMIZE
)

m.optimize()

# # Print solution (change later, may be wrong)
# print("Optimal objective value:", m.objVal)
# print("Train stop patterns:")
# for t in range(TRAIN_COUNT):
#     stops = [j for j in range(STATION_COUNT) if x[t][j].X > 0.5]
#     print(f"Train {t}: {stops}")

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 12th Gen Intel(R) Core(TM) i5-12450H, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 7407 rows, 3670 columns and 25105 nonzeros (Min)
Model fingerprint: 0x1c93beb2
Model has 3635 linear objective coefficients
Variable types: 35 continuous, 3635 integer (3635 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+02]
  Objective range  [1e+00, 1e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+03]

Found heuristic solution: objective 5300.0000000
Presolve removed 3973 rows and 237 columns
Presolve time: 0.08s
Presolved: 3434 rows, 3433 columns, 13433 nonzeros
Variable types: 0 continuous, 3433 integer (3433 binary)

Root relaxation: objective 3.500000e+03, 100 iterations, 0.01 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl U

## Model ends here for now. below information is not included, may or may not be required.

### Placeholder

## Capacities for each arc
* Should be the same for every connected line, because 1 train will traverse the entire line of the same color.
* if the arc is describing a tranfer arc, i.e. via manual walking e.g. jurong east green to red line, we can assume the capacities to be infinite (capacities here in this case refer to the number of people that can transfer from 1 line to another, which is technically much larger when transiting between lines than the fixed capacity of a single train transporting passengers for a single line)
* referencing this article: https://medium.com/@simpletan/deep-dive-of-singapores-rail-network-passenger-loads-e331d3b9626b

In [ ]:
capacities = {}

# Supply and demand at each node
* Distance-defined. To be simulated based on distance from CBD
* Considering only evening peak hour, so supply (i.e. people going into the MRT) is higher as we get closer to the various CBD locations.

In [ ]:
supply = {
    1: GRB.INFINITY,
    2: 0,
    3: 0,
    4: 0
}
# just an example of supply. change later

# Create the model

In [ ]:
m = gp.model('minimize_cost_to_travel')

# Constraints

In [ ]:
# flow = m.addVars(arcs,ub=[capacities[arc] for arc in arcs])
# selling_amount = m.addVar(name="selling_amount")

# # Flow conservation constraints at nodes 
# for i in supply:
#     if (i == 1):
#         m.addConstr(flow.sum(i, '*') == selling_amount)
#     elif (i == 5):
#         m.addConstr(flow.sum('*', i) == selling_amount)
#     else:
#         m.addConstr(flow.sum(i, '*') - flow.sum('*', i) == 0)


# Objective function: 
* Requires 3 inputs from the user, depending on how much the user values each item. we will ask for 3 weights, and all 3 weights must sum up to 100
* All weights must also be positive.

w1 * total travel time + w2 * transfer penalties + w3 * overcrowding penalties (utility penalty)

In [ ]:
while (True):
    w1 = int(input("On a scale of 1-100, how much do you value total travel time? "))
    w2 = int(input("On a scale of 1-100, how much do you value number of transfers? "))
    w3 = int(input("on a scale of 1-100, how much do you value overcrowding? "))
    if (w1 + w2 + w3 == 100):
        break
    else:
        print("The weights must sum to 100. Please try again.")

# Set objective
# m.setObjective(w1*travelling_time + w2*number_of_transfers + w3*overcrowding, GRB.MINIMIZE)


# Optimizing the model
* print the solutions also, which will show the entire route to take from station A to station B.

In [ ]:
m.optimize()


# # Print the solution
# print("---------------------------------")
# print("Optimal flow and cost:")
# print(selling_amount.VarName, selling_amount)
# print(f"Total profit: {m.objVal}")